## Math Dataset (Countdown Tasks)


In [ ]:
import os
from datasets import load_dataset

dataset = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", cache_dir="/mnt/lustre/koa/scratch/kantas/hf_home/datasets")

print(f"Train examples: {len(dataset)}")

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/490364 [00:00<?, ? examples/s]

Train examples: 1


## Math Baseline Evaluation (Qwen2.5-Math-1.5B, r1_zero prompt, GSM8k test)

In [ ]:
import json

results_path = "output/Qwen_Math_1.5B_gsm8k_r1zero_results.json"
with open(results_path) as f:
    results = json.load(f)

# Categorize each example
correct     = [r for r in results if r["format_reward"] == 1.0 and r["answer_reward"] == 1.0]
format_only = [r for r in results if r["format_reward"] == 1.0 and r["answer_reward"] == 0.0]
no_format   = [r for r in results if r["format_reward"] == 0.0]

total = len(results)
print(f"Total examples: {total}")
print()
print(f"(1) Correct      (format=1, answer=1): {len(correct):5d}  ({100*len(correct)/total:.1f}%)")
print(f"(2) Format only  (format=1, answer=0): {len(format_only):5d}  ({100*len(format_only)/total:.1f}%)")
print(f"(3) No format    (format=0, answer=0): {len(no_format):5d}  ({100*len(no_format)/total:.1f}%)")

### Cases where format reward = 0 (no `</think> <answer>...</answer>` structure detected)

In [ ]:
import random

sample_no_format = random.sample(no_format, min(10, len(no_format)))

for i, r in enumerate(sample_no_format, 1):
    gen = r["model_generation"]
    has_think_close = "</think>" in gen
    has_answer_open = "<answer>" in gen
    has_answer_close = "</answer>" in gen
    # Check the specific conjunction the reward fn requires
    has_required_junction = "</think> <answer>" in gen

    print(f"--- Example {i} ---")
    print(f"Question: {r['question'][:120]}")
    print(f"Ground truth: {r['ground_truth']}")
    print(f"Has </think>: {has_think_close} | Has <answer>: {has_answer_open} | Has </answer>: {has_answer_close} | Has '</think> <answer>': {has_required_junction}")
    print(f"Generation tail (last 300 chars):\n  {repr(gen[-300:])}")
    print()

**Analysis — format reward = 0:**

The reward function in `drgrpo_grader.py` checks for the *exact* substring `"</think> <answer>"` (with a single space between the two tags). There are two plausible failure modes:

1. **Output truncation** — the model hit `max_new_tokens=2048` before closing its reasoning, so `</think>` was never emitted. Look at the generation tail: if it ends mid-sentence the model simply ran out of budget.

2. **Spacing/format mismatch** — the model emitted `</think>\n<answer>` or `</think><answer>` (no space, or a newline instead of a space). The reward function does *not* use a regex; it requires the literal string `"</think> <answer>"`. If `</think>` is present but the junction is missing, this is a parser strictness issue, not a model failure.

Check the flags above: if `has_think_close=True` but `has_required_junction=False`, the issue is the **parser** (strict space requirement). If `has_think_close=False`, the issue is the **output** (truncation or the model never closed its thinking block).

### Cases where format reward = 1 but answer reward = 0 (wrong answer despite correct structure)

In [ ]:
import re

sample_format_only = random.sample(format_only, min(10, len(format_only)))

for i, r in enumerate(sample_format_only, 1):
    gen = r["model_generation"]
    full_response = "<think>" + gen

    # Extract what the model put in <answer>...</answer>
    answer_match = re.search(r"<answer>(.*?)</answer>", full_response, re.DOTALL)
    extracted_answer = answer_match.group(1).strip() if answer_match else "(none)"

    print(f"--- Example {i} ---")
    print(f"Question: {r['question'][:120]}")
    print(f"Ground truth:      {r['ground_truth']}")
    print(f"Extracted answer:  {extracted_answer}")
    print()

**Analysis — format reward = 1, answer reward = 0:**

These cases formatted correctly but got the math wrong. Common failure patterns to look for:

- **Arithmetic slip** — the reasoning chain is mostly right but makes a calculation error partway through, propagating a wrong final value.
- **Wrong answer extracted** — the model put a `\boxed{}` expression or an intermediate result in `<answer>` instead of the final numeric answer. The grader calls `extract_answer` on anything with `\boxed` inside the answer tag, so a misplaced boxed expression can cause a mismatch.
- **Unit or representation mismatch** — the grader normalizes well but may miss edge cases like `$18.00` vs `18` or a fraction vs decimal if the normalization chain doesn't cover it.
- **Genuinely wrong reasoning** — the 1.5B model simply lacks the arithmetic capability for multi-step word problems without RLVR training.

In [ ]:
import json, glob
points = []
for path in glob.glob("../models/*/training_log.json"):
    log = json.load(open(path))
    meta = next(e for e in log if e.get('meta'))
    acc  = next(e for e in log if e.get('step') == 'final_vllm')
    points.append((meta['num_examples'], acc['val_accuracy']))
points.sort(key=lambda x: x[0] if x[0] != 'full' else 9999)